# GPU vs CPU Accuracy and Performance Comparison
This notebook compares the GPU-accelerated Poisson solver against the original CPU implementation. It focuses on the uniform grid case (`azu_unif=2`, `rad_unif=1`) as the nonuniform GPU components are not yet implemented.

We will compare:
1. **Accuracy**: The L2 relative error of the GPU solver vs. the CPU solver.
2. **Performance**: The speedup achieved by using the GPU.

In [6]:
import os, sys
import pandas as pd
import cupy as cp

# --- Environment Setup ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  # If in Colab, clone the public repository from GitHub
  repo_name = 'NUFFTRR_Poisson'
  repo_url = 'https://github.com/charles-d-burton/NUFFTRR_Poisson.git'
  repo_root = f'/content/{repo_name}'

  if not os.path.exists(repo_root):
    !git clone {repo_url}
  else:
    print('Repository already exists. Pulling latest changes...')
    !cd {repo_root} && git pull
else:
  # Use the local path if not in Colab
  repo_root = r"C:\Users\charl\OneDrive\Documents\GitHub\NUFFTRR_Poisson"

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from Tests.CPUvsGPU.helpers import (
    run_comparison_pipeline,
    render_multitable,
    _prepare_table2_df
)

print(f"CuPy is using GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")

Cloning into 'NUFFTRR_Poisson'...
fatal: could not read Username for 'https://github.com': No such device or address


FileNotFoundError: [Errno 2] No such file or directory: '/content/NUFFTRR_Poisson'

# Parameters

In [ ]:
N_vals = [32, 64, 128, 256, 512]
M_vals = [32, 64, 128, 256, 512]
N_fixed = 32

MUTE_OUTPUT = True

# Run Comparison

In [ ]:
print("Running comparison for varying N and M values...")
df_nm = run_comparison_pipeline(N_vals, M_vals, test_type="VaryingNM", mute=MUTE_OUTPUT)

print("\nRunning comparison for varying M, BC, and Quadrature Rule (N fixed at 32)...")
df_bc_quad = run_comparison_pipeline(None, M_vals, test_type="VaryingM_BC_Quad", fixed_val=N_fixed, mute=MUTE_OUTPUT)

print("\nDone.")

# Results: Varying N and M

In [ ]:
render_multitable(
    df_nm,
    index_col="N",
    columns_col="M",
    value_cols=["L2_rel_cpu", "L2_rel_gpu", "accuracy_diff", "runtime_cpu", "runtime_gpu", "speedup"],
    titles=["CPU Accuracy (L2 Rel. Error)", "GPU Accuracy (L2 Rel. Error)", "Accuracy Difference (Abs)", 
            "CPU Runtime (s)", "GPU Runtime (s)", "Speedup (CPU/GPU)"],
    float_formats=[
        lambda x: f"{x:.2e}", lambda x: f"{x:.2e}", lambda x: f"{x:.2e}",
        lambda x: f"{x:.4f}", lambda x: f"{x:.4f}", lambda x: f"{x:.2f}x"
    ]
)

# Results: Varying M, BC, and Quadrature

In [ ]:
df_bc_quad_fmt = _prepare_table2_df(df_bc_quad)
render_multitable(
    df_bc_quad_fmt,
    index_col="M",
    columns_col=["quad_str", "bc_str"],
    value_cols=["L2_rel_cpu", "L2_rel_gpu", "accuracy_diff", "runtime_cpu", "runtime_gpu", "speedup"],
    titles=["CPU Accuracy (L2 Rel. Error)", "GPU Accuracy (L2 Rel. Error)", "Accuracy Difference (Abs)", 
            "CPU Runtime (s)", "GPU Runtime (s)", "Speedup (CPU/GPU)"],
    float_formats=[
        lambda x: f"{x:.2e}", lambda x: f"{x:.2e}", lambda x: f"{x:.2e}",
        lambda x: f"{x:.4f}", lambda x: f"{x:.4f}", lambda x: f"{x:.2f}x"
    ]
)